# Data Drift and Model Drift Monitoring with MLflow

## Topic Goal


In production, a model may perform well during training but later degrade because:

1. Incoming data distribution changes.
2. Customer behavior changes.
3. Feature patterns change.
4. Target distribution changes.
5. Prediction distribution changes.
6. Model performance on current labeled data drops.

This notebook demonstrates an end-to-end drift monitoring workflow:

1. Load baseline training data.
2. Load current/new data.
3. Train and evaluate a baseline model.
4. Score the current dataset using the trained model.
5. Compare baseline and current feature distributions.
6. Calculate numeric drift using PSI.
7. Calculate categorical drift using PSI.
8. Compare baseline vs current model performance when labels are available.
9. Detect prediction drift.
10. Create drift status flags.
11. Save drift reports and charts.
12. Log all drift artifacts and metrics to MLflow.
13. Log model with signature and input example in the **same run**.
14. Register the model using the same-run `model_uri`.

This notebook is designed for experienced professionals learning production-style MLOps monitoring.


## 1. Import Libraries

We use only common Python libraries:

- `pandas` and `numpy` for drift calculations
- `matplotlib` for drift charts
- `scikit-learn` for model training and evaluation
- `mlflow` for tracking, model logging, artifacts, and registry


In [ ]:
# Import os to create folders and manage paths.
import os

# Import json to save drift metadata reports.
import json

# Import warnings to keep output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import datetime to record monitoring timestamp.
from datetime import datetime

# Import MLflow for tracking, model logging, and registry.
import mlflow

# Import sklearn flavor for logging sklearn pipelines.
import mlflow.sklearn

# Import infer_signature for MLflow model signature.
from mlflow.models.signature import infer_signature

# Import pandas for tabular data processing.
import pandas as pd

# Import numpy for numerical calculations.
import numpy as np

# Import matplotlib for drift visualizations.
import matplotlib.pyplot as plt

# Import train_test_split for creating baseline train/test split.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for preprocessing numerical and categorical columns.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical variables.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical variables.
from sklearn.preprocessing import StandardScaler

# Import Pipeline to combine preprocessing and model.
from sklearn.pipeline import Pipeline

# Import RandomForestClassifier for churn prediction.
from sklearn.ensemble import RandomForestClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Configure MLflow and Folders

This notebook uses local file-based MLflow tracking.

The folder structure expected is:

```text
datasets/customer_churn_500.csv
datasets/customer_churn_current_500.csv
outputs/
artifacts/
mlruns/
```

The current dataset is used to simulate production/new incoming data.


In [ ]:
# Define MLflow experiment name.
EXPERIMENT_NAME = "advanced_15_data_model_drift"

# Define MLflow run name.
RUN_NAME = "15_data_model_drift_same_run_usecase"

# Remove inherited MLflow Project experiment ID if present.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited MLflow run ID if present.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create MLflow local tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create outputs folder for drift reports and charts.
os.makedirs("outputs", exist_ok=True)

# Create artifacts folder for any supporting files.
os.makedirs("artifacts", exist_ok=True)

# Configure local MLflow tracking URI.
mlflow.set_tracking_uri("file:./mlruns")

# Define baseline dataset path.
BASELINE_DATA_PATH = "datasets/customer_churn_500.csv"

# Define current/new dataset path.
CURRENT_DATA_PATH = "datasets/customer_churn_current_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Print configuration.
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Baseline dataset:", BASELINE_DATA_PATH)
print("Current dataset:", CURRENT_DATA_PATH)
print("Tracking URI:", mlflow.get_tracking_uri())


## 3. Load Baseline and Current Datasets

The baseline dataset represents training/reference data.

The current dataset represents new/production data.

If the current dataset is unavailable, the notebook creates a simple shifted copy from the baseline dataset so the notebook remains executable.


In [ ]:
# Check baseline dataset.
if not os.path.exists(BASELINE_DATA_PATH):
    # Stop clearly if baseline dataset is missing.
    raise FileNotFoundError(f"Baseline dataset not found: {BASELINE_DATA_PATH}")

# Load baseline dataset.
baseline_df = pd.read_csv(BASELINE_DATA_PATH)

# Check whether current dataset exists.
if os.path.exists(CURRENT_DATA_PATH):
    # Load current/new dataset.
    current_df = pd.read_csv(CURRENT_DATA_PATH)

    # Track current dataset source.
    current_dataset_source = CURRENT_DATA_PATH
else:
    # Create a fallback shifted current dataset.
    current_df = baseline_df.copy()

    # Increase monthly charges to simulate data drift.
    current_df["monthly_charges"] = current_df["monthly_charges"] * 1.15

    # Increase support tickets to simulate operational issue.
    current_df["support_tickets"] = current_df["support_tickets"] + 1

    # Track fallback source.
    current_dataset_source = "generated_shifted_copy_from_baseline"

# Display baseline sample.
display(baseline_df.head())

# Display current sample.
display(current_df.head())

# Print shapes.
print("Baseline shape:", baseline_df.shape)
print("Current shape:", current_df.shape)
print("Current dataset source:", current_dataset_source)


## 4. Define Drift Helper Functions

We use **Population Stability Index (PSI)** for drift detection.

PSI interpretation commonly used in industry:

```text
PSI < 0.10       -> no major drift
0.10 to 0.25     -> moderate drift
PSI >= 0.25      -> significant drift
```

This notebook implements PSI for both numerical and categorical features.


In [ ]:
# Define a small value to avoid division by zero.
EPSILON = 1e-6

# Define PSI threshold for moderate drift.
PSI_MODERATE_THRESHOLD = 0.10

# Define PSI threshold for significant drift.
PSI_SIGNIFICANT_THRESHOLD = 0.25

# Define performance drop threshold.
PERFORMANCE_DROP_THRESHOLD = 0.05


# Define function to calculate PSI from expected and actual percentage arrays.
def calculate_psi_from_percentages(expected_percents, actual_percents):
    # Convert inputs to numpy arrays.
    expected_percents = np.array(expected_percents) + EPSILON

    # Convert actual values to numpy array.
    actual_percents = np.array(actual_percents) + EPSILON

    # Calculate PSI value.
    psi_value = np.sum((actual_percents - expected_percents) * np.log(actual_percents / expected_percents))

    # Return PSI as float.
    return float(psi_value)


# Define function to calculate numerical PSI.
def calculate_numeric_psi(baseline_series, current_series, bins=10):
    # Drop missing values from baseline.
    baseline_clean = baseline_series.dropna()

    # Drop missing values from current.
    current_clean = current_series.dropna()

    # Create quantile-based bins from baseline data.
    _, bin_edges = pd.qcut(
        baseline_clean,
        q=bins,
        retbins=True,
        duplicates="drop"
    )

    # Ensure first bin captures lower values.
    bin_edges[0] = -np.inf

    # Ensure last bin captures higher values.
    bin_edges[-1] = np.inf

    # Bin baseline values.
    baseline_bins = pd.cut(baseline_clean, bins=bin_edges)

    # Bin current values using same baseline bin edges.
    current_bins = pd.cut(current_clean, bins=bin_edges)

    # Calculate baseline distribution.
    baseline_dist = baseline_bins.value_counts(normalize=True, sort=False)

    # Calculate current distribution.
    current_dist = current_bins.value_counts(normalize=True, sort=False)

    # Align distributions.
    baseline_dist, current_dist = baseline_dist.align(current_dist, fill_value=0)

    # Calculate PSI.
    psi_value = calculate_psi_from_percentages(baseline_dist.values, current_dist.values)

    # Return PSI.
    return psi_value


# Define function to calculate categorical PSI.
def calculate_categorical_psi(baseline_series, current_series):
    # Convert baseline to string and fill missing values.
    baseline_clean = baseline_series.fillna("MISSING").astype(str)

    # Convert current to string and fill missing values.
    current_clean = current_series.fillna("MISSING").astype(str)

    # Calculate baseline category distribution.
    baseline_dist = baseline_clean.value_counts(normalize=True)

    # Calculate current category distribution.
    current_dist = current_clean.value_counts(normalize=True)

    # Align categories.
    baseline_dist, current_dist = baseline_dist.align(current_dist, fill_value=0)

    # Calculate PSI.
    psi_value = calculate_psi_from_percentages(baseline_dist.values, current_dist.values)

    # Return PSI.
    return psi_value


# Define drift severity function.
def classify_drift(psi_value):
    # Significant drift condition.
    if psi_value >= PSI_SIGNIFICANT_THRESHOLD:
        return "significant_drift"

    # Moderate drift condition.
    if psi_value >= PSI_MODERATE_THRESHOLD:
        return "moderate_drift"

    # No major drift condition.
    return "no_major_drift"


## 5. Prepare Baseline Training Data

We train the model on the baseline dataset.

The target column is:

```text
churn
```


In [ ]:
# Define target column.
target_column = "churn"

# Define ID column.
id_column = "customer_id"

# Create baseline feature matrix.
X = baseline_df.drop(columns=[id_column, target_column])

# Create baseline target vector.
y = baseline_df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split baseline data into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print feature groups.
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 6. Train Baseline Model and Evaluate on Baseline Test Data

This gives us reference performance.

Later we compare this with performance on current data if current labels are available.


In [ ]:
# Create Random Forest model.
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    random_state=42
)

# Create complete sklearn pipeline.
pipeline = Pipeline(steps=[
    # Apply preprocessing first.
    ("preprocessor", preprocessor),

    # Train model after preprocessing.
    ("model", model),
])

# Train the pipeline.
pipeline.fit(X_train, y_train)

# Predict baseline test data.
baseline_test_predictions = pipeline.predict(X_test)

# Calculate baseline accuracy.
baseline_accuracy = accuracy_score(y_test, baseline_test_predictions)

# Calculate baseline precision.
baseline_precision = precision_score(y_test, baseline_test_predictions, zero_division=0)

# Calculate baseline recall.
baseline_recall = recall_score(y_test, baseline_test_predictions, zero_division=0)

# Calculate baseline F1-score.
baseline_f1 = f1_score(y_test, baseline_test_predictions, zero_division=0)

# Create baseline classification report.
baseline_report = classification_report(y_test, baseline_test_predictions, zero_division=0)

# Save baseline report.
with open("outputs/baseline_classification_report.txt", "w") as file:
    file.write(baseline_report)

# Print baseline metrics.
print("Baseline accuracy:", baseline_accuracy)
print("Baseline precision:", baseline_precision)
print("Baseline recall:", baseline_recall)
print("Baseline F1-score:", baseline_f1)
print(baseline_report)


## 7. Data Drift Detection: Feature Distribution Drift

This section compares baseline feature distributions with current feature distributions.

We calculate PSI for:

- numerical features
- categorical features

The output is saved as:

```text
outputs/data_drift_report.csv
```


In [ ]:
# Prepare current features by dropping ID and target when present.
current_features = current_df.drop(columns=[id_column, target_column], errors="ignore")

# Create empty list for drift rows.
drift_rows = []

# Calculate numerical feature drift.
for feature in numerical_columns:
    # Calculate numerical PSI.
    psi_value = calculate_numeric_psi(X[feature], current_features[feature])

    # Append drift result.
    drift_rows.append({
        "feature": feature,
        "feature_type": "numerical",
        "psi": psi_value,
        "drift_status": classify_drift(psi_value),
        "baseline_mean": X[feature].mean(),
        "current_mean": current_features[feature].mean(),
        "mean_difference": current_features[feature].mean() - X[feature].mean(),
    })

# Calculate categorical feature drift.
for feature in categorical_columns:
    # Calculate categorical PSI.
    psi_value = calculate_categorical_psi(X[feature], current_features[feature])

    # Append drift result.
    drift_rows.append({
        "feature": feature,
        "feature_type": "categorical",
        "psi": psi_value,
        "drift_status": classify_drift(psi_value),
        "baseline_mean": np.nan,
        "current_mean": np.nan,
        "mean_difference": np.nan,
    })

# Create drift report dataframe.
data_drift_report = pd.DataFrame(drift_rows)

# Sort by PSI descending.
data_drift_report = data_drift_report.sort_values("psi", ascending=False)

# Save drift report.
data_drift_report.to_csv("outputs/data_drift_report.csv", index=False)

# Display drift report.
display(data_drift_report)


## 8. Data Drift Visualization

We create a chart showing top drifting features by PSI.

This artifact is logged to MLflow.


In [ ]:
# Select top drifting features.
top_drift_features = data_drift_report.head(10)

# Create PSI bar chart.
plt.figure(figsize=(10, 6))
plt.barh(top_drift_features["feature"], top_drift_features["psi"])
plt.gca().invert_yaxis()
plt.axvline(PSI_MODERATE_THRESHOLD, linestyle="--", label="Moderate drift threshold")
plt.axvline(PSI_SIGNIFICANT_THRESHOLD, linestyle="--", label="Significant drift threshold")
plt.xlabel("PSI")
plt.title("Top Feature Drift by PSI")
plt.legend()
plt.tight_layout()

# Save chart.
plt.savefig("outputs/top_feature_drift_psi.png")
plt.close()

# Print file path.
print("Saved chart: outputs/top_feature_drift_psi.png")


## 9. Prediction Drift Detection

Prediction drift checks whether the model's prediction distribution changes between baseline and current data.

This is useful even when current labels are not available.


In [ ]:
# Predict on baseline full feature dataset.
baseline_full_predictions = pipeline.predict(X)

# Predict on current feature dataset.
current_predictions = pipeline.predict(current_features)

# Calculate prediction distribution for baseline.
baseline_prediction_distribution = pd.Series(baseline_full_predictions).value_counts(normalize=True).sort_index()

# Calculate prediction distribution for current.
current_prediction_distribution = pd.Series(current_predictions).value_counts(normalize=True).sort_index()

# Align distributions.
baseline_prediction_distribution, current_prediction_distribution = baseline_prediction_distribution.align(
    current_prediction_distribution,
    fill_value=0
)

# Calculate prediction PSI.
prediction_psi = calculate_psi_from_percentages(
    baseline_prediction_distribution.values,
    current_prediction_distribution.values
)

# Classify prediction drift.
prediction_drift_status = classify_drift(prediction_psi)

# Create prediction drift dataframe.
prediction_drift_report = pd.DataFrame({
    "class_label": baseline_prediction_distribution.index,
    "baseline_prediction_percent": baseline_prediction_distribution.values,
    "current_prediction_percent": current_prediction_distribution.values,
})

# Save prediction drift report.
prediction_drift_report.to_csv("outputs/prediction_drift_report.csv", index=False)

# Display prediction drift report.
display(prediction_drift_report)

# Print prediction drift summary.
print("Prediction PSI:", prediction_psi)
print("Prediction drift status:", prediction_drift_status)


## 10. Model Drift Detection: Performance on Current Labeled Data

Model drift means model performance has changed over time.

If the current dataset has actual labels, we can compare current performance with baseline performance.

This notebook checks current performance because the synthetic current dataset contains `churn`.


In [ ]:
# Check whether current labels are available.
current_labels_available = target_column in current_df.columns

# Initialize current performance metrics.
current_accuracy = None
current_precision = None
current_recall = None
current_f1 = None
f1_drop = None
model_drift_status = "not_evaluated_no_current_labels"

# Evaluate current performance if labels are available.
if current_labels_available:
    # Extract current target.
    current_y = current_df[target_column]

    # Calculate current accuracy.
    current_accuracy = accuracy_score(current_y, current_predictions)

    # Calculate current precision.
    current_precision = precision_score(current_y, current_predictions, zero_division=0)

    # Calculate current recall.
    current_recall = recall_score(current_y, current_predictions, zero_division=0)

    # Calculate current F1-score.
    current_f1 = f1_score(current_y, current_predictions, zero_division=0)

    # Calculate F1 performance drop.
    f1_drop = baseline_f1 - current_f1

    # Decide model drift status based on F1 drop.
    model_drift_status = "model_drift_detected" if f1_drop >= PERFORMANCE_DROP_THRESHOLD else "no_major_model_drift"

    # Create current classification report.
    current_report = classification_report(current_y, current_predictions, zero_division=0)

    # Save current report.
    with open("outputs/current_classification_report.txt", "w") as file:
        file.write(current_report)

    # Print current metrics.
    print("Current accuracy:", current_accuracy)
    print("Current precision:", current_precision)
    print("Current recall:", current_recall)
    print("Current F1-score:", current_f1)
    print("F1 drop:", f1_drop)
    print("Model drift status:", model_drift_status)
    print(current_report)
else:
    # Print message when labels are unavailable.
    print("Current labels are not available. Model performance drift cannot be directly evaluated.")


## 11. Overall Drift Summary

This section combines data drift, prediction drift, and model drift into a single summary.


In [ ]:
# Count significant feature drift.
significant_feature_drift_count = int((data_drift_report["drift_status"] == "significant_drift").sum())

# Count moderate feature drift.
moderate_feature_drift_count = int((data_drift_report["drift_status"] == "moderate_drift").sum())

# Calculate maximum PSI.
max_feature_psi = float(data_drift_report["psi"].max())

# Decide overall data drift status.
overall_data_drift_status = (
    "significant_data_drift"
    if significant_feature_drift_count > 0
    else "moderate_data_drift"
    if moderate_feature_drift_count > 0
    else "no_major_data_drift"
)

# Create monitoring summary dictionary.
drift_summary = {
    "monitoring_timestamp": datetime.now().isoformat(),
    "baseline_dataset": BASELINE_DATA_PATH,
    "current_dataset": current_dataset_source,
    "psi_moderate_threshold": PSI_MODERATE_THRESHOLD,
    "psi_significant_threshold": PSI_SIGNIFICANT_THRESHOLD,
    "performance_drop_threshold": PERFORMANCE_DROP_THRESHOLD,
    "max_feature_psi": max_feature_psi,
    "moderate_feature_drift_count": moderate_feature_drift_count,
    "significant_feature_drift_count": significant_feature_drift_count,
    "overall_data_drift_status": overall_data_drift_status,
    "prediction_psi": float(prediction_psi),
    "prediction_drift_status": prediction_drift_status,
    "current_labels_available": current_labels_available,
    "baseline_f1": float(baseline_f1),
    "current_f1": None if current_f1 is None else float(current_f1),
    "f1_drop": None if f1_drop is None else float(f1_drop),
    "model_drift_status": model_drift_status,
}

# Save drift summary JSON.
with open("outputs/drift_summary.json", "w") as file:
    json.dump(drift_summary, file, indent=4)

# Create summary dataframe.
drift_summary_df = pd.DataFrame([drift_summary])

# Save summary CSV.
drift_summary_df.to_csv("outputs/drift_summary.csv", index=False)

# Display summary.
display(drift_summary_df)


## 12. Infer Model Signature

The registered model should still include a proper input/output signature.

This is done using baseline test input.


In [ ]:
# Create input example for signature.
input_example = X_test.head(5)

# Generate sample predictions for signature.
sample_predictions = pipeline.predict(input_example)

# Infer signature.
signature = infer_signature(input_example, sample_predictions)

# Display input example.
display(input_example)

# Print sample predictions.
print("Sample predictions:", sample_predictions)


## 13. Same-Run MLflow Logging with Drift Artifacts

This is the main production-style MLflow run.

Inside the **same run**, we log:

- baseline model metrics
- current model metrics
- data drift metrics
- prediction drift metrics
- model drift metrics
- drift reports
- drift charts
- model with signature and input example
- model URI

Then we register the model using the same `model_uri`.


In [ ]:
# Start same MLflow run.
with mlflow.start_run(run_name=RUN_NAME):

    # Log dataset paths.
    mlflow.log_param("baseline_dataset", BASELINE_DATA_PATH)
    mlflow.log_param("current_dataset", current_dataset_source)

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Log PSI thresholds.
    mlflow.log_param("psi_moderate_threshold", PSI_MODERATE_THRESHOLD)
    mlflow.log_param("psi_significant_threshold", PSI_SIGNIFICANT_THRESHOLD)

    # Log performance drop threshold.
    mlflow.log_param("performance_drop_threshold", PERFORMANCE_DROP_THRESHOLD)

    # Log baseline metrics.
    mlflow.log_metric("baseline_accuracy", baseline_accuracy)
    mlflow.log_metric("baseline_precision", baseline_precision)
    mlflow.log_metric("baseline_recall", baseline_recall)
    mlflow.log_metric("baseline_f1_score", baseline_f1)

    # Log current metrics only when labels are available.
    if current_labels_available:
        mlflow.log_metric("current_accuracy", current_accuracy)
        mlflow.log_metric("current_precision", current_precision)
        mlflow.log_metric("current_recall", current_recall)
        mlflow.log_metric("current_f1_score", current_f1)
        mlflow.log_metric("f1_drop", f1_drop)

    # Log drift metrics.
    mlflow.log_metric("max_feature_psi", max_feature_psi)
    mlflow.log_metric("moderate_feature_drift_count", moderate_feature_drift_count)
    mlflow.log_metric("significant_feature_drift_count", significant_feature_drift_count)
    mlflow.log_metric("prediction_psi", prediction_psi)

    # Set drift status tags.
    mlflow.set_tag("overall_data_drift_status", overall_data_drift_status)
    mlflow.set_tag("prediction_drift_status", prediction_drift_status)
    mlflow.set_tag("model_drift_status", model_drift_status)
    mlflow.set_tag("run_purpose", "data_and_model_drift_monitoring")

    # Log reports and charts.
    mlflow.log_artifact("outputs/baseline_classification_report.txt")
    mlflow.log_artifact("outputs/data_drift_report.csv")
    mlflow.log_artifact("outputs/top_feature_drift_psi.png")
    mlflow.log_artifact("outputs/prediction_drift_report.csv")
    mlflow.log_artifact("outputs/drift_summary.json")
    mlflow.log_artifact("outputs/drift_summary.csv")

    # Log current classification report if available.
    if os.path.exists("outputs/current_classification_report.txt"):
        mlflow.log_artifact("outputs/current_classification_report.txt")

    # Log model with signature and input example in the SAME run.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Register model using same-run model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print final details.
print("Same-run model URI:", model_uri)
print("Registered model name:", registered_model.name)
print("Registered model version:", registered_model.version)




- Data drift means input feature distributions have changed.
- Prediction drift means the model is producing different prediction distributions.
- Model drift means the model's real performance has dropped.
- Data drift can be detected even without labels.
- Model drift needs actual labels from current data.
- PSI is a common practical metric for drift monitoring.
- Drift monitoring results should be logged as metrics, tags, and artifacts.
- Drift charts and reports help teams decide whether retraining is needed.
- The model, signature, drift metrics, and reports are logged in one traceable MLflow run.
